# Auto MPG Regression Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('auto-mpg.csv')
print('First 10 rows:'); print(df.head(10).to_string())
print('Shape:', df.shape)


In [ ]:
df['horsepower'] = df['horsepower'].replace('?', np.nan)
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
print('Missing values:\n', df.isna().sum())
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
if 'car name' in df.columns:
    df = df.drop(columns=['car name'])
print('After cleaning shape:', df.shape)


In [ ]:
print(df.describe())


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, col in zip(axes, ['weight','horsepower','displacement']):
    ax.scatter(df[col], df['mpg'], alpha=0.6)
    ax.set_xlabel(col); ax.set_ylabel('mpg'); ax.set_title(f'{col} vs mpg')
plt.tight_layout(); plt.show()


In [ ]:
sns.pairplot(df[['mpg','weight','horsepower','displacement','acceleration','origin']], hue='origin')
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap'); plt.show()
print('Correlation with mpg:\n', df.corr(numeric_only=True)['mpg'].sort_values())


In [ ]:
X = df.drop(columns=['mpg'])
y = df['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Train:', X_train_s.shape, 'Test:', X_test_s.shape)


In [ ]:
configs = [
    ('Linear', SVR(kernel='linear', C=1)),
    ('RBF C=1', SVR(kernel='rbf', C=1)),
    ('RBF C=10', SVR(kernel='rbf', C=10)),
    ('Poly deg2 C=1', SVR(kernel='poly', degree=2, C=1)),
]
results=[]
for name, model in configs:
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results.append({'Model':name,'MAE':mae,'RMSE':rmse,'R2':r2})
    print(name, 'R2=', r2)
res_df = pd.DataFrame(results)
print(res_df.to_string())


In [ ]:
best = res_df.loc[res_df['R2'].idxmax(),'Model']
print('Best:', best)
best_model = dict(configs)[best]
pred = best_model.predict(X_test_s)
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],'r--')
plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title(f'Predicted vs Actual ({best})'); plt.show()
residuals = y_test - pred
plt.figure(figsize=(6,4))
plt.scatter(pred, residuals, alpha=0.6)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted'); plt.ylabel('Residuals'); plt.title('Residuals vs Predicted'); plt.show()
